# 07. Results and boundary validation

Interprets the terminally corrected policy results, decomposes VDO by product and week, and validates the asynchronous policy-boundary mechanism on a fine funding grid. The earlier unmatched support-sensitivity branch is intentionally excluded from the final pipeline.

## 1. Load the frozen policy artifact

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Mapping, Sequence
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        root
        for root in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (root / "pyproject.toml").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. Run from the repository or notebooks directory."
    )

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import price_of_extrapolation.policy as policy
from price_of_extrapolation.policy import (
    build_schedule_system,
    load_pickle,
    save_pickle,
    schedule_input_fingerprint,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
POLICY_ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "policy"
CACHE_DIR = PROJECT_ROOT / "artifacts" / "cache" / "policy"
TABLE_DIR = PROJECT_ROOT / "results" / "final" / "tables"
FIGURE_DIR = PROJECT_ROOT / "results" / "final" / "figures"

for directory in [POLICY_ARTIFACT_DIR, CACHE_DIR, TABLE_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

POLICY_ARTIFACT_PATH = POLICY_ARTIFACT_DIR / "policy_optimization.pkl"
if not POLICY_ARTIFACT_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {POLICY_ARTIFACT_PATH}. Run Notebook 06 first."
    )

artifact = load_pickle(POLICY_ARTIFACT_PATH)
policy_results = (
    artifact["policy_results"]
    .copy()
    .sort_values(["capacity", "alpha"])
    .reset_index(drop=True)
)
product_decomposition = artifact["product_decomposition"].copy()
weekly_decomposition = artifact["weekly_decomposition"].copy()
schedules = artifact["schedules"]
draws_by_product = artifact["draws_by_product"]
weekly_profiles = artifact["weekly_profiles"]
action_sets = artifact["action_sets"]
support_table = artifact["support_table"]
products = list(artifact["products"])
planning = artifact["schedule_system"]["planning"]

MAIN_ALPHA = float(artifact["main_alpha"])
SELECTED_WASHOUT = int(artifact["selected_washout"])
ECONOMIC_PROFILE_MODE = str(artifact["economic_profile_mode"])

if not all(np.allclose(np.asarray(v["price_factor"], dtype=float), 1.0) for v in weekly_profiles.values()):
    raise AssertionError("Price factors are not fixed to one in the final artifact.")
if not all(np.allclose(np.asarray(v["cost_factor"], dtype=float), 1.0) for v in weekly_profiles.values()):
    raise AssertionError("Cost factors are not fixed to one in the final artifact.")

print("Selected washout:", SELECTED_WASHOUT)
print("Main alpha:", MAIN_ALPHA)
print("Policy rows:", len(policy_results))


## 2. Locate policy boundaries

In [ ]:
policy_regimes = (
    policy_results.copy()
)

for signature_column in [
    "dynamic_schedule_signature",
    "myopic_schedule_signature",
    "dynamic_active_set",
    "myopic_active_set",
]:
    policy_regimes[
        f"{signature_column}_previous"
    ] = (
        policy_regimes.groupby(
            "capacity",
            observed=True,
        )[
            signature_column
        ].shift()
    )

policy_regimes[
    "dynamic_schedule_change"
] = (
    policy_regimes[
        "dynamic_schedule_signature"
    ]
    != policy_regimes[
        "dynamic_schedule_signature_previous"
    ]
)
policy_regimes[
    "myopic_schedule_change"
] = (
    policy_regimes[
        "myopic_schedule_signature"
    ]
    != policy_regimes[
        "myopic_schedule_signature_previous"
    ]
)
policy_regimes[
    "dynamic_active_set_change"
] = (
    policy_regimes[
        "dynamic_active_set"
    ]
    != policy_regimes[
        "dynamic_active_set_previous"
    ]
)
policy_regimes[
    "myopic_active_set_change"
] = (
    policy_regimes[
        "myopic_active_set"
    ]
    != policy_regimes[
        "myopic_active_set_previous"
    ]
)

first_rows = (
    policy_regimes.groupby(
        "capacity",
        observed=True,
    ).cumcount().eq(0)
)

for column in [
    "dynamic_schedule_change",
    "myopic_schedule_change",
    "dynamic_active_set_change",
    "myopic_active_set_change",
]:
    policy_regimes.loc[
        first_rows,
        column,
    ] = False

policy_regimes[
    "any_policy_boundary"
] = (
    policy_regimes[
        [
            "dynamic_schedule_change",
            "myopic_schedule_change",
            "dynamic_active_set_change",
            "myopic_active_set_change",
        ]
    ].any(axis=1)
)

policy_regimes[
    "relative_best_second_gap"
] = (
    policy_regimes[
        "best_second_gap"
    ]
    / policy_regimes[
        "dynamic_profit"
    ].abs()
)

display(
    policy_regimes.loc[
        policy_regimes[
            "any_policy_boundary"
        ]
    ][
        [
            "alpha",
            "capacity",
            "vdo",
            "vdo_percent",
            "best_second_gap",
            "dynamic_schedule_change",
            "myopic_schedule_change",
            "dynamic_active_set_change",
            "myopic_active_set_change",
        ]
    ].head(40)
)

In [ ]:
top_vdo_by_capacity = (
    policy_regimes.loc[
        policy_regimes.groupby(
            "capacity",
            observed=True,
        )[
            "vdo"
        ].idxmax()
    ][
        [
            "capacity",
            "alpha",
            "vdo",
            "vdo_percent",
            "best_second_gap",
            "action_disagreements",
            "status_disagreements",
            "depth_disagreements",
            "timing_shift_products",
        ]
    ]
    .sort_values("capacity")
    .reset_index(drop=True)
)

display(top_vdo_by_capacity)

## 3. Main policy-regime figure

In [ ]:
fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(12.6, 4.8),
    gridspec_kw={
        "width_ratios": [1.05, 0.95],
        "wspace": 0.30,
    },
)

for capacity, group in (
    policy_regimes.groupby(
        "capacity",
        observed=True,
    )
):
    axes[0].plot(
        group["alpha"],
        group["vdo"],
        label=f"B={capacity}",
    )

    boundary_rows = group.loc[
        group["any_policy_boundary"]
    ]
    axes[0].scatter(
        boundary_rows["alpha"],
        boundary_rows["vdo"],
        s=18,
    )

axes[0].axhline(
    0.0,
    linestyle=":",
    linewidth=1.0,
)
axes[0].axvline(
    MAIN_ALPHA,
    linestyle="--",
    linewidth=1.0,
)
axes[0].set_xlabel(
    r"Contract generosity, $\alpha$"
)
axes[0].set_ylabel(
    "Value of dynamic optimization"
)
axes[0].set_title(
    "(a) Dynamic value and policy boundaries"
)
axes[0].legend(
    title="Weekly capacity"
)

for capacity, group in (
    policy_regimes.groupby(
        "capacity",
        observed=True,
    )
):
    axes[1].plot(
        group["alpha"],
        100.0 * group["disagreement_rate"],
        label=f"B={capacity}",
    )

axes[1].axvline(
    MAIN_ALPHA,
    linestyle="--",
    linewidth=1.0,
)
axes[1].set_xlabel(
    r"Contract generosity, $\alpha$"
)
axes[1].set_ylabel(
    "Dynamic-myopic action disagreement (%)"
)
axes[1].set_title(
    "(b) Policy disagreement"
)

fig.tight_layout()

POLICY_REGIME_FIGURE_PNG = (
    FIGURE_DIR
    / "figure_02_policy_regimes_and_disagreement.png"
)
POLICY_REGIME_FIGURE_PDF = (
    FIGURE_DIR
    / "figure_02_policy_regimes_and_disagreement.pdf"
)

fig.savefig(
    POLICY_REGIME_FIGURE_PNG,
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    POLICY_REGIME_FIGURE_PDF,
    bbox_inches="tight",
)

plt.show()

Dots in Panel (a) mark funding values at which at least one dynamic or myopic schedule or active set changes relative to the preceding grid point. They should be interpreted as discrete policy boundaries, not statistical significance markers.

## 4. Exact product and week decomposition

In [ ]:
MAIN_CAPACITY = 2

main_alpha_available = float(
    policy_results.loc[
        policy_results["capacity"].eq(
            MAIN_CAPACITY
        ),
        "alpha",
    ].iloc[
        np.argmin(
            np.abs(
                policy_results.loc[
                    policy_results[
                        "capacity"
                    ].eq(
                        MAIN_CAPACITY
                    ),
                    "alpha",
                ].to_numpy()
                - MAIN_ALPHA
            )
        )
    ]
)

boundary_row = (
    policy_results.loc[
        policy_results[
            "capacity"
        ].eq(
            MAIN_CAPACITY
        )
    ]
    .sort_values(
        "vdo",
        ascending=False,
    )
    .iloc[0]
)
BOUNDARY_ALPHA = float(
    boundary_row["alpha"]
)

DECOMPOSITION_ALPHAS = sorted(
    {
        round(
            main_alpha_available,
            8,
        ),
        round(
            BOUNDARY_ALPHA,
            8,
        ),
    }
)

print(
    "Decomposition alphas:",
    DECOMPOSITION_ALPHAS,
)

In [ ]:
selected_product_decomposition = (
    product_decomposition.loc[
        product_decomposition[
            "capacity"
        ].eq(
            MAIN_CAPACITY
        )
        & product_decomposition[
            "alpha"
        ].round(8).isin(
            DECOMPOSITION_ALPHAS
        )
    ]
    .copy()
)

selected_weekly_decomposition = (
    weekly_decomposition.loc[
        weekly_decomposition[
            "capacity"
        ].eq(
            MAIN_CAPACITY
        )
        & weekly_decomposition[
            "alpha"
        ].round(8).isin(
            DECOMPOSITION_ALPHAS
        )
    ]
    .copy()
)

product_accounting_check = (
    selected_product_decomposition.groupby(
        "alpha",
        observed=True,
    )[
        "vdo_contribution"
    ].sum()
)

weekly_accounting_check = (
    selected_weekly_decomposition.groupby(
        "alpha",
        observed=True,
    )[
        "vdo_contribution"
    ].sum()
)

display(
    pd.concat(
        [
            product_accounting_check.rename(
                "product_sum"
            ),
            weekly_accounting_check.rename(
                "weekly_sum"
            ),
        ],
        axis=1,
    )
)

In [ ]:
product_rankings = (
    selected_product_decomposition.sort_values(
        [
            "alpha",
            "vdo_contribution",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

display(
    product_rankings[
        [
            "alpha",
            "upc",
            "dynamic_profit",
            "myopic_profit",
            "vdo_contribution",
        ]
    ]
)

washout_attribution = (
    selected_weekly_decomposition.groupby(
        [
            "alpha",
            "decision_week",
        ],
        observed=True,
    )[
        "vdo_contribution"
    ]
    .sum()
    .rename(
        "vdo_contribution"
    )
    .reset_index()
)

display(washout_attribution)

In [ ]:
fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(12.4, 4.8),
    gridspec_kw={
        "width_ratios": [1.0, 1.0],
        "wspace": 0.30,
    },
)

for alpha, group in (
    selected_weekly_decomposition.groupby(
        "alpha",
        observed=True,
    )
):
    axes[0].plot(
        group["week"],
        group["cumulative_vdo"],
        marker="o",
        markersize=3,
        label=rf"$\alpha={alpha:.2f}$",
    )

axes[0].axvline(
    artifact[
        "schedule_system"
    ][
        "planning"
    ].decision_horizon,
    linestyle="--",
    linewidth=1.0,
)
axes[0].axhline(
    0.0,
    linestyle=":",
    linewidth=1.0,
)
axes[0].set_xlabel(
    "Evaluation week"
)
axes[0].set_ylabel(
    "Cumulative VDO"
)
axes[0].set_title(
    "(a) When the dynamic gain is realized"
)
axes[0].legend()

main_product = (
    selected_product_decomposition.loc[
        np.isclose(
            selected_product_decomposition[
                "alpha"
            ],
            main_alpha_available,
        )
    ]
    .sort_values(
        "vdo_contribution"
    )
)

axes[1].barh(
    main_product["upc"].astype(str),
    main_product["vdo_contribution"],
)
axes[1].axvline(
    0.0,
    linestyle=":",
    linewidth=1.0,
)
axes[1].set_xlabel(
    "Product contribution to VDO"
)
axes[1].set_ylabel(
    "UPC"
)
axes[1].set_title(
    rf"(b) Product accounting at $\alpha={main_alpha_available:.2f}$"
)

fig.tight_layout()

DECOMPOSITION_FIGURE_PNG = (
    FIGURE_DIR
    / "figure_03_vdo_decomposition.png"
)
DECOMPOSITION_FIGURE_PDF = (
    FIGURE_DIR
    / "figure_03_vdo_decomposition.pdf"
)

fig.savefig(
    DECOMPOSITION_FIGURE_PNG,
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    DECOMPOSITION_FIGURE_PDF,
    bbox_inches="tight",
)

plt.show()

The product and week decompositions are additive accounting identities:
\[
\operatorname{VDO}
=
\sum_i\left[V_i(\pi^D)-V_i(\pi^M)\right]
=
\sum_t\left[V_t(\pi^D)-V_t(\pi^M)\right].
\]
Because products interact through the weekly capacity constraint, these contributions should not be interpreted as independent causal treatment effects.

## 5. Largest adjacent policy switches

In [ ]:
def changed_products(
    previous_schedule: dict[str, np.ndarray],
    current_schedule: dict[str, np.ndarray],
) -> tuple[list[str], int]:
    products = sorted(
        set(previous_schedule)
        | set(current_schedule)
    )

    changed = []
    changed_cells = 0

    for upc in products:
        previous = np.asarray(
            previous_schedule[upc],
            dtype=float,
        )
        current = np.asarray(
            current_schedule[upc],
            dtype=float,
        )

        differences = ~np.isclose(
            previous,
            current,
        )

        if differences.any():
            changed.append(upc)
            changed_cells += int(
                differences.sum()
            )

    return changed, changed_cells


switch_rows = []

for capacity, group in (
    policy_results.groupby(
        "capacity",
        observed=True,
    )
):
    group = group.sort_values(
        "alpha"
    )

    previous_row = None

    for row in group.itertuples(
        index=False
    ):
        current_key = (
            round(
                float(row.alpha),
                8,
            ),
            int(capacity),
        )

        if previous_row is not None:
            previous_key = (
                round(
                    float(
                        previous_row.alpha
                    ),
                    8,
                ),
                int(capacity),
            )

            dynamic_changed, dynamic_cells = changed_products(
                schedules[
                    previous_key
                ][
                    "dynamic"
                ],
                schedules[
                    current_key
                ][
                    "dynamic"
                ],
            )
            myopic_changed, myopic_cells = changed_products(
                schedules[
                    previous_key
                ][
                    "myopic"
                ],
                schedules[
                    current_key
                ][
                    "myopic"
                ],
            )

            if (
                dynamic_cells > 0
                or myopic_cells > 0
            ):
                switch_rows.append(
                    {
                        "capacity": (
                            capacity
                        ),
                        "alpha_previous": (
                            previous_row.alpha
                        ),
                        "alpha_current": (
                            row.alpha
                        ),
                        "vdo_previous": (
                            previous_row.vdo
                        ),
                        "vdo_current": (
                            row.vdo
                        ),
                        "vdo_change": (
                            row.vdo
                            - previous_row.vdo
                        ),
                        "dynamic_changed_products": (
                            "|".join(
                                dynamic_changed
                            )
                        ),
                        "dynamic_changed_cells": (
                            dynamic_cells
                        ),
                        "myopic_changed_products": (
                            "|".join(
                                myopic_changed
                            )
                        ),
                        "myopic_changed_cells": (
                            myopic_cells
                        ),
                    }
                )

        previous_row = row

policy_switches = (
    pd.DataFrame(
        switch_rows
    )
    .sort_values(
        "vdo_change",
        key=lambda values: (
            values.abs()
        ),
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    policy_switches.head(20)
)

## 6. Fine-grid validation of the asynchronous boundary mechanism

The main interior boundary is re-evaluated for capacity $B=2$ on a 0.01 funding grid. Candidate schedules are rebuilt because pruning depends on the evaluated funding range.

In [ ]:
# Focused validation design.
VALIDATION_CAPACITY = 2
ALPHA_MIN = 2.40
ALPHA_MAX = 2.65
ALPHA_STEP = 0.01

FINE_ALPHA_GRID = np.round(
    np.arange(
        ALPHA_MIN,
        ALPHA_MAX + 0.5 * ALPHA_STEP,
        ALPHA_STEP,
    ),
    4,
)

SCHEDULE_BATCH_SIZE = 256
MILP_TIME_LIMIT_SECONDS = None
COMPUTE_SECOND_BEST = True

print("Capacity:", VALIDATION_CAPACITY)
print(
    "Fine alpha grid:",
    FINE_ALPHA_GRID[0],
    "to",
    FINE_ALPHA_GRID[-1],
    f"({len(FINE_ALPHA_GRID)} values)",
)

In [ ]:
expected_fingerprint = (
    schedule_input_fingerprint(
        draws_by_product=draws_by_product,
        weekly_profiles=weekly_profiles,
        action_sets=action_sets,
        planning=planning,
        alpha_grid=FINE_ALPHA_GRID,
    )
)

cache_path = (
    CACHE_DIR
    / (
        "local_boundary_schedule_system_"
        f"b{VALIDATION_CAPACITY}_"
        f"a{ALPHA_MIN:.2f}_{ALPHA_MAX:.2f}_"
        f"s{ALPHA_STEP:.2f}_"
        f"w{SELECTED_WASHOUT}.pkl"
    )
)

local_schedule_system = None

if cache_path.is_file():
    cached = load_pickle(cache_path)

    if (
        cached.get("input_fingerprint")
        == expected_fingerprint
    ):
        local_schedule_system = cached
        print("Loaded current cache:", cache_path.name)

if local_schedule_system is None:
    local_schedule_system = (
        build_schedule_system(
            draws_by_product=draws_by_product,
            weekly_profiles=weekly_profiles,
            action_sets=action_sets,
            planning=planning,
            alpha_grid=FINE_ALPHA_GRID,
            batch_size=SCHEDULE_BATCH_SIZE,
        )
    )

    save_pickle(
        local_schedule_system,
        cache_path,
    )

    print("Built schedule system:", cache_path.name)

In [ ]:
summary_rows = []
product_frames = []
weekly_frames = []
local_schedules = {}

for alpha in FINE_ALPHA_GRID:
    alpha = float(alpha)

    dynamic_solution = (
        policy.solve_dynamic_category(
            schedule_system=local_schedule_system,
            alpha=alpha,
            capacity=VALIDATION_CAPACITY,
            compute_second_best=COMPUTE_SECOND_BEST,
            time_limit_seconds=MILP_TIME_LIMIT_SECONDS,
        )
    )

    myopic_solution = (
        policy.simulate_myopic_category(
            draws_by_product=draws_by_product,
            weekly_profiles=weekly_profiles,
            action_sets=action_sets,
            planning=planning,
            alpha=alpha,
            capacity=VALIDATION_CAPACITY,
        )
    )

    (
        summary,
        product_decomposition,
        weekly_decomposition,
    ) = policy.analyze_policy_pair(
        dynamic_solution=dynamic_solution,
        myopic_solution=myopic_solution,
        draws_by_product=draws_by_product,
        weekly_profiles=weekly_profiles,
        planning=planning,
        support_table=support_table,
        alpha=alpha,
        capacity=VALIDATION_CAPACITY,
    )

    summary["solver_message"] = (
        dynamic_solution["solver_message"]
    )
    summary[
        "economic_profile_mode"
    ] = ECONOMIC_PROFILE_MODE

    summary_rows.append(summary)
    product_frames.append(
        product_decomposition
    )
    weekly_frames.append(
        weekly_decomposition
    )

    local_schedules[
        (
            round(alpha, 8),
            VALIDATION_CAPACITY,
        )
    ] = {
        "dynamic": dynamic_solution[
            "schedule_map"
        ],
        "myopic": myopic_solution[
            "schedule_map"
        ],
    }

local_results = (
    pd.DataFrame(summary_rows)
    .sort_values("alpha")
    .reset_index(drop=True)
)

local_product_decomposition = (
    pd.concat(
        product_frames,
        ignore_index=True,
    )
)

local_weekly_decomposition = (
    pd.concat(
        weekly_frames,
        ignore_index=True,
    )
)

if (
    local_results["vdo"] < -1e-6
).any():
    raise AssertionError(
        "Dynamic profit falls below myopic profit "
        "at one or more fine-grid values."
    )

maximum_vdo_error = float(
    (
        local_results["vdo"]
        - (
            local_results["dynamic_profit"]
            - local_results["myopic_profit"]
        )
    ).abs().max()
)

if maximum_vdo_error > 1e-6:
    raise AssertionError(
        "VDO arithmetic check failed. "
        f"Maximum error: {maximum_vdo_error}"
    )

print("Fine-grid validation passed.")
print("Solver messages:")
for message in local_results[
    "solver_message"
].drop_duplicates():
    print(" -", message)

display(
    local_results[
        [
            "alpha",
            "dynamic_profit",
            "myopic_profit",
            "vdo",
            "vdo_percent",
            "dynamic_promotion_count",
            "myopic_promotion_count",
            "action_disagreements",
            "best_second_gap",
        ]
    ]
)

In [ ]:
coarse_capacity_results = (
    policy_results.loc[
        policy_results[
            "capacity"
        ].eq(
            VALIDATION_CAPACITY
        )
    ]
    .copy()
)

overlap_alphas = sorted(
    set(
        np.round(
            local_results["alpha"],
            8,
        )
    )
    .intersection(
        set(
            np.round(
                coarse_capacity_results[
                    "alpha"
                ],
                8,
            )
        )
    )
)

overlap_check = (
    local_results.loc[
        local_results[
            "alpha"
        ].round(8).isin(
            overlap_alphas
        ),
        [
            "alpha",
            "dynamic_profit",
            "myopic_profit",
            "vdo",
        ],
    ]
    .merge(
        coarse_capacity_results.loc[
            coarse_capacity_results[
                "alpha"
            ].round(8).isin(
                overlap_alphas
            ),
            [
                "alpha",
                "dynamic_profit",
                "myopic_profit",
                "vdo",
            ],
        ],
        on="alpha",
        suffixes=(
            "_fine",
            "_coarse",
        ),
        validate="one_to_one",
    )
)

for column in [
    "dynamic_profit",
    "myopic_profit",
    "vdo",
]:
    overlap_check[
        f"{column}_difference"
    ] = (
        overlap_check[
            f"{column}_fine"
        ]
        - overlap_check[
            f"{column}_coarse"
        ]
    )

difference_columns = [
    column
    for column in overlap_check.columns
    if column.endswith("_difference")
]

maximum_overlap_difference = float(
    overlap_check[
        difference_columns
    ].abs().max().max()
)

print(
    "Maximum fine-versus-coarse difference:",
    maximum_overlap_difference,
)

if maximum_overlap_difference > 1e-5:
    raise AssertionError(
        "The fine-grid run does not reproduce "
        "Notebook 06 at overlapping alpha values."
    )

display(
    overlap_check.round(8)
)

In [ ]:
def compare_schedule_maps(
    previous: Mapping[
        str,
        Sequence[float],
    ],
    current: Mapping[
        str,
        Sequence[float],
    ],
) -> dict[str, object]:
    all_products = sorted(
        set(previous)
        | set(current)
    )

    changed_products = []
    changed_cells = 0
    status_changes = 0
    depth_changes = 0

    for upc in all_products:
        previous_values = np.asarray(
            previous[upc],
            dtype=float,
        )
        current_values = np.asarray(
            current[upc],
            dtype=float,
        )

        changed = ~np.isclose(
            previous_values,
            current_values,
        )

        if changed.any():
            changed_products.append(
                str(upc)
            )
            changed_cells += int(
                changed.sum()
            )

        previous_positive = (
            previous_values > 0
        )
        current_positive = (
            current_values > 0
        )

        status_changes += int(
            np.sum(
                previous_positive
                != current_positive
            )
        )

        depth_changes += int(
            np.sum(
                previous_positive
                & current_positive
                & changed
            )
        )

    return {
        "changed_products": "|".join(
            changed_products
        ),
        "changed_product_count": len(
            changed_products
        ),
        "changed_cells": changed_cells,
        "status_changes": status_changes,
        "depth_changes": depth_changes,
    }


local_results = local_results.sort_values(
    "alpha"
).reset_index(drop=True)

local_results[
    "dynamic_boundary"
] = (
    local_results[
        "dynamic_schedule_signature"
    ]
    .ne(
        local_results[
            "dynamic_schedule_signature"
        ].shift()
    )
)
local_results[
    "myopic_boundary"
] = (
    local_results[
        "myopic_schedule_signature"
    ]
    .ne(
        local_results[
            "myopic_schedule_signature"
        ].shift()
    )
)

local_results.loc[
    0,
    [
        "dynamic_boundary",
        "myopic_boundary",
    ],
] = False

transition_rows = []

for position in range(
    1,
    len(local_results),
):
    previous_row = local_results.iloc[
        position - 1
    ]
    current_row = local_results.iloc[
        position
    ]

    if not (
        bool(
            current_row[
                "dynamic_boundary"
            ]
        )
        or bool(
            current_row[
                "myopic_boundary"
            ]
        )
    ):
        continue

    previous_key = (
        round(
            float(
                previous_row["alpha"]
            ),
            8,
        ),
        VALIDATION_CAPACITY,
    )
    current_key = (
        round(
            float(
                current_row["alpha"]
            ),
            8,
        ),
        VALIDATION_CAPACITY,
    )

    dynamic_change = compare_schedule_maps(
        local_schedules[
            previous_key
        ][
            "dynamic"
        ],
        local_schedules[
            current_key
        ][
            "dynamic"
        ],
    )

    myopic_change = compare_schedule_maps(
        local_schedules[
            previous_key
        ][
            "myopic"
        ],
        local_schedules[
            current_key
        ][
            "myopic"
        ],
    )

    dynamic_boundary = bool(
        current_row[
            "dynamic_boundary"
        ]
    )
    myopic_boundary = bool(
        current_row[
            "myopic_boundary"
        ]
    )

    if (
        dynamic_boundary
        and myopic_boundary
    ):
        boundary_type = "both"
    elif dynamic_boundary:
        boundary_type = "dynamic_only"
    else:
        boundary_type = "myopic_only"

    transition_rows.append(
        {
            "capacity": (
                VALIDATION_CAPACITY
            ),
            "alpha_previous": float(
                previous_row["alpha"]
            ),
            "alpha_current": float(
                current_row["alpha"]
            ),
            "boundary_type": (
                boundary_type
            ),
            "vdo_previous": float(
                previous_row["vdo"]
            ),
            "vdo_current": float(
                current_row["vdo"]
            ),
            "vdo_change": float(
                current_row["vdo"]
                - previous_row["vdo"]
            ),
            "vdo_percent_current": float(
                current_row[
                    "vdo_percent"
                ]
            ),
            "dynamic_promotion_count_previous": int(
                previous_row[
                    "dynamic_promotion_count"
                ]
            ),
            "dynamic_promotion_count_current": int(
                current_row[
                    "dynamic_promotion_count"
                ]
            ),
            "myopic_promotion_count_previous": int(
                previous_row[
                    "myopic_promotion_count"
                ]
            ),
            "myopic_promotion_count_current": int(
                current_row[
                    "myopic_promotion_count"
                ]
            ),
            "best_second_gap_current": float(
                current_row[
                    "best_second_gap"
                ]
            ),
            **{
                f"dynamic_{key}": value
                for key, value in (
                    dynamic_change.items()
                )
            },
            **{
                f"myopic_{key}": value
                for key, value in (
                    myopic_change.items()
                )
            },
        }
    )

boundary_transitions = (
    pd.DataFrame(
        transition_rows
    )
)

if not boundary_transitions.empty:
    boundary_transitions[
        "absolute_vdo_change"
    ] = (
        boundary_transitions[
            "vdo_change"
        ].abs()
    )

    boundary_transitions = (
        boundary_transitions
        .sort_values(
            "absolute_vdo_change",
            ascending=False,
        )
        .reset_index(drop=True)
    )

print(
    "Detected local boundaries:",
    len(boundary_transitions),
)

display(
    boundary_transitions
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(9.4, 5.2)
)

ax.plot(
    local_results["alpha"],
    local_results["vdo_percent"],
    marker="o",
    markersize=4,
    linewidth=1.5,
    label="VDO",
)

dynamic_boundaries = (
    local_results.loc[
        local_results[
            "dynamic_boundary"
        ]
    ]
)
myopic_boundaries = (
    local_results.loc[
        local_results[
            "myopic_boundary"
        ]
    ]
)

ax.scatter(
    dynamic_boundaries["alpha"],
    dynamic_boundaries[
        "vdo_percent"
    ],
    marker="^",
    s=60,
    label="Dynamic schedule switch",
)

ax.scatter(
    myopic_boundaries["alpha"],
    myopic_boundaries[
        "vdo_percent"
    ],
    marker="s",
    s=44,
    label="Myopic schedule switch",
)

ax.axhline(
    0.0,
    linestyle=":",
    linewidth=1.0,
)

ax.set_xlabel(
    r"Contract generosity, $\alpha$"
)
ax.set_ylabel(
    "VDO relative to myopic profit (%)"
)
ax.set_title(
    "Local validation of the policy-boundary mechanism "
    f"(B={VALIDATION_CAPACITY})"
)
ax.legend()

fig.tight_layout()

BOUNDARY_FIGURE_PNG = (
    FIGURE_DIR
    / "appendix_local_boundary_vdo.png"
)
BOUNDARY_FIGURE_PDF = (
    FIGURE_DIR
    / "appendix_local_boundary_vdo.pdf"
)

fig.savefig(
    BOUNDARY_FIGURE_PNG,
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    BOUNDARY_FIGURE_PDF,
    bbox_inches="tight",
)

plt.show()

In [ ]:
fig, ax = plt.subplots(
    figsize=(9.4, 4.8)
)

ax.plot(
    local_results["alpha"],
    local_results[
        "best_second_gap"
    ],
    marker="o",
    markersize=4,
    linewidth=1.5,
)

for alpha in dynamic_boundaries[
    "alpha"
]:
    ax.axvline(
        alpha,
        linestyle="--",
        linewidth=0.8,
    )

ax.set_xlabel(
    r"Contract generosity, $\alpha$"
)
ax.set_ylabel(
    "Best minus second-best dynamic value"
)
ax.set_title(
    "Local dynamic-schedule separation"
)

fig.tight_layout()

GAP_FIGURE_PNG = (
    FIGURE_DIR
    / "appendix_local_boundary_schedule_gap.png"
)
GAP_FIGURE_PDF = (
    FIGURE_DIR
    / "appendix_local_boundary_schedule_gap.pdf"
)

fig.savefig(
    GAP_FIGURE_PNG,
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    GAP_FIGURE_PDF,
    bbox_inches="tight",
)

plt.show()

## 7. Save final diagnostics

In [ ]:
# Save publication-facing diagnostics and the combined results artifact.
output_tables = {
    "policy_regimes.csv": policy_regimes,
    "top_vdo_by_capacity.csv": top_vdo_by_capacity,
    "product_vdo_decomposition.csv": selected_product_decomposition,
    "weekly_vdo_decomposition.csv": selected_weekly_decomposition,
    "policy_boundary_switches.csv": policy_switches,
    "local_boundary_results.csv": local_results,
    "local_boundary_transitions.csv": boundary_transitions,
    "local_boundary_product_decomposition.csv": local_product_decomposition,
    "local_boundary_weekly_decomposition.csv": local_weekly_decomposition,
}

for filename, frame in output_tables.items():
    frame.to_csv(TABLE_DIR / filename, index=False)

RESULTS_ARTIFACT_PATH = POLICY_ARTIFACT_DIR / "results_analysis.pkl"
results_artifact = {
    "policy_regimes": policy_regimes,
    "top_vdo_by_capacity": top_vdo_by_capacity,
    "decomposition_alphas": DECOMPOSITION_ALPHAS,
    "product_decomposition": selected_product_decomposition,
    "weekly_decomposition": selected_weekly_decomposition,
    "policy_switches": policy_switches,
    "local_boundary_results": local_results,
    "local_boundary_transitions": boundary_transitions,
    "local_boundary_schedules": local_schedules,
    "local_boundary_product_decomposition": local_product_decomposition,
    "local_boundary_weekly_decomposition": local_weekly_decomposition,
    "overlap_check": overlap_check,
    "economic_profile_mode": ECONOMIC_PROFILE_MODE,
    "selected_washout": SELECTED_WASHOUT,
}
save_pickle(results_artifact, RESULTS_ARTIFACT_PATH)

print("Saved:", RESULTS_ARTIFACT_PATH)


## Interpretation guardrails

- VDO is a model-implied counterfactual comparison, not a causal treatment effect.
- Sharp changes in VDO are attributed to asynchronous discrete schedule switches only when the fine-grid transition table confirms the mechanism.
- A small best-minus-second-best gap indicates schedule fragility; it is not a confidence interval.
- Product and week contributions are exact accounting decompositions, but category-capacity interactions prevent independent causal interpretation.
- Support sensitivity is omitted from the final results until the baseline action grid can be reproduced exactly under every threshold rule.